### Завдання 1.
Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних.

In [3]:
import os
import urllib.request
from datetime import datetime
import pandas as pd  
folder = "vhi_data"
if not os.path.exists(folder):
    os.makedirs(folder)
def download_vhi_data(province_id):
    existing_files = [f for f in os.listdir(folder) if f.startswith(f"vhi_id_{province_id}_")]
    if existing_files:
        print(f"Дані для області {province_id} вже існують. Пропускаємо...")
        return
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    now = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"vhi_id_{province_id}_{now}.csv"
    filepath = os.path.join(folder, filename)
    try:
        urllib.request.urlretrieve(url, filepath)
        print(f"Завантажено: {filename}")
    except Exception as e:
        print(f"Помилка при завантаженні області {province_id}: {e}")
for i in range(1, 28):
    download_vhi_data(i)
print("Завантаження завершено!")

Завантажено: vhi_id_1_20260313_193820.csv
Завантажено: vhi_id_2_20260313_193821.csv
Завантажено: vhi_id_3_20260313_193822.csv
Завантажено: vhi_id_4_20260313_193823.csv
Завантажено: vhi_id_5_20260313_193824.csv
Завантажено: vhi_id_6_20260313_193825.csv
Завантажено: vhi_id_7_20260313_193826.csv
Завантажено: vhi_id_8_20260313_193827.csv
Завантажено: vhi_id_9_20260313_193828.csv
Завантажено: vhi_id_10_20260313_193829.csv
Завантажено: vhi_id_11_20260313_193830.csv
Завантажено: vhi_id_12_20260313_193831.csv
Завантажено: vhi_id_13_20260313_193832.csv
Завантажено: vhi_id_14_20260313_193833.csv
Завантажено: vhi_id_15_20260313_193834.csv
Завантажено: vhi_id_16_20260313_193834.csv
Завантажено: vhi_id_17_20260313_193836.csv
Завантажено: vhi_id_18_20260313_193838.csv
Завантажено: vhi_id_19_20260313_193838.csv
Завантажено: vhi_id_20_20260313_193839.csv
Завантажено: vhi_id_21_20260313_193840.csv
Завантажено: vhi_id_22_20260313_193841.csv
Завантажено: vhi_id_23_20260313_193843.csv
Завантажено: vhi_id_

### Завдання 2.
Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області.

In [13]:
def read_and_clean_vhi_data(directory_path='vhi_data'):
    files = glob.glob(f"{directory_path}/*.csv")
    dataframes = []
    headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
    province_names = {
        1: 'Черкаська', 2: 'Чернігівська', 3: 'Чернівецька', 4: 'Республіка Крим', 
        5: 'Дніпропетровська', 6: 'Донецька', 7: 'Івано-Франківська', 8: 'Харківська', 
        9: 'Херсонська', 10: 'Хмельницька', 11: 'Київська', 12: 'м. Київ', 
        13: 'Кіровоградська', 14: 'Луганська', 15: 'Львівська', 16: 'Миколаївська', 
        17: 'Одеська', 18: 'Полтавська', 19: 'Рівненська', 20: 'м. Севастополь', 
        21: 'Сумська', 22: 'Тернопільська', 23: 'Закарпатська', 24: 'Вінницька', 
        25: 'Волинська', 26: 'Запорізька', 27: 'Житомирська'
    }
    for file in files:
        province_id = int(file.split('vhi_id_')[1].split('_')[0])
        df = pd.read_csv(file, header=1, names=headers, skipinitialspace=True)
        df.drop(columns=['empty'], inplace=True, errors='ignore')
        df.dropna(subset=['VHI'], inplace=True)
        df['Year'] = df['Year'].astype(str).str.replace(r'<tt><pre>', '', regex=True)
        df = df[df['Year'] != '']
        df['Year'] = df['Year'].astype(int)
        df['Old_Province_ID'] = province_id
        df['Province_Name'] = province_names[province_id]
        dataframes.append(df)
    combined_dataframe = pd.concat(dataframes, ignore_index=True)
    return combined_dataframe
df_vhi = read_and_clean_vhi_data()
print("Загальна кількість записів:")
print(len(df_vhi))
display(df_vhi.head())

Загальна кількість записів:
60372


,Year,Week,SMN,SMT,VCI,TCI,VHI,Old_Province_ID,Province_Name
0,1982,1.0,0.059,258.24,51.11,48.78,49.95,10,Хмельницька
1,1982,2.0,0.063,261.53,55.89,38.20,47.04,10,Хмельницька
2,1982,3.0,0.063,263.45,57.30,32.69,44.99,10,Хмельницька
3,1982,4.0,0.061,265.10,53.96,28.62,41.29,10,Хмельницька
4,1982,5.0,0.058,266.42,46.87,28.57,37.72,10,Хмельницька


### Завдання 3.
Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалися за українською абеткою (1 область - Вінницька).

In [17]:
def update_province_indices(df):
    ukr_alphabet_order = [
        'Вінницька', 'Волинська', 'Дніпропетровська', 'Донецька', 'Житомирська',
        'Закарпатська', 'Запорізька', 'Івано-Франківська', 'Київська', 'Кіровоградська',
        'Луганська', 'Львівська', 'Миколаївська', 'Одеська', 'Полтавська',
        'Рівненська', 'Сумська', 'Тернопільська', 'Харківська', 'Херсонська',
        'Хмельницька', 'Черкаська', 'Чернівецька', 'Чернігівська', 
        'Республіка Крим', 'м. Київ', 'м. Севастополь'
    ]
    new_index_mapping = {province: idx for idx, province in enumerate(ukr_alphabet_order, start=1)}
    df_updated = df.copy()
    df_updated['Province_ID'] = df_updated['Province_Name'].map(new_index_mapping)
    df_updated = df_updated.drop(columns=['Old_Province_ID'], errors='ignore')
    return df_updated
df_vhi = update_province_indices(df_vhi)
print("Дані після зміни індексів:")
display(df_vhi[df_vhi['Province_ID'] == 1].head())

Дані після зміни індексів:


,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_Name,Province_ID
33540,1982,1.0,0.068,263.59,63.47,28.34,45.90,Вінницька,1
33541,1982,2.0,0.074,265.78,67.62,23.05,45.34,Вінницька,1
33542,1982,3.0,0.076,267.19,69.37,20.40,44.88,Вінницька,1
33543,1982,4.0,0.075,268.57,65.26,17.93,41.60,Вінницька,1
33544,1982,5.0,0.072,269.24,58.58,20.00,39.29,Вінницька,1


### Завдання 4. 
Реалізувати процедури для формування вибірок.
#### Завдання 4.1.
Ряд VHI для області за вказаний рік.

In [18]:
def get_vhi_by_year_and_province(df, province_id, year):
    result = df[(df['Province_ID'] == province_id) & (df['Year'] == year)]
    return result[['Year', 'Week', 'VHI', 'Province_Name']]
vhi_2010_vinnytsia = get_vhi_by_year_and_province(df_vhi, province_id=1, year=2010)
print("Ряд VHI для Вінницької області за 2010 рік:")
display(vhi_2010_vinnytsia.head(10)) 

Ряд VHI для Вінницької області за 2010 рік:


,Year,Week,VHI,Province_Name
34996,2010,1.0,52.04,Вінницька
34997,2010,2.0,51.05,Вінницька
34998,2010,3.0,52.37,Вінницька
34999,2010,4.0,52.69,Вінницька
35000,2010,5.0,53.16,Вінницька
35001,2010,6.0,53.20,Вінницька
35002,2010,7.0,53.00,Вінницька
35003,2010,8.0,53.48,Вінницька
35004,2010,9.0,53.69,Вінницька
35005,2010,10.0,51.78,Вінницька


### Завдання 4.2. 
Ряд VHI за вказаний діапазон років для вказаних областей.

In [23]:
def get_vhi_by_years_and_provinces(df, province_ids, start_year, end_year):
    result = df[
        (df['Province_ID'].isin(province_ids)) & 
        (df['Year'] >= start_year) & 
        (df['Year'] <= end_year)
    ]
    return result[['Year', 'Week', 'VHI', 'Province_Name']]
vhi_range = get_vhi_by_years_and_provinces(df_vhi, province_ids=[1, 5], start_year=2010, end_year=2019)
print("Ряд VHI для Вінницької та Житомирської областей за 2010-2019 роки:")
display(vhi_range.sample(10))

Ряд VHI для Вінницької та Житомирської областей за 2010-2019 роки:


,Year,Week,VHI,Province_Name
41711,2010,8.0,47.17,Житомирська
41884,2013,25.0,54.38,Житомирська
41885,2013,26.0,53.79,Житомирська
41737,2010,34.0,46.68,Житомирська
35256,2015,1.0,47.22,Вінницька
42075,2017,8.0,40.26,Житомирська
35141,2012,42.0,31.60,Вінницька
41718,2010,15.0,36.52,Житомирська
42154,2018,35.0,51.82,Житомирська
41784,2011,29.0,56.27,Житомирська


### Завдання 4.3. 
Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани.

In [25]:
def get_vhi_statistics(df, province_id, years):
    subset = df[(df['Province_ID'] == province_id) & (df['Year'].isin(years))]
    if subset.empty:
        print("Немає даних для вказаних параметрів.")
        return pd.DataFrame()
    province_name = subset['Province_Name'].iloc[0]
    print(f"--- Порівняльна статистика VHI для області '{province_name}' ---")
    stats = subset.groupby('Year')['VHI'].agg(['min', 'max', 'mean', 'median']).round(2)
    stats.columns = ['Мінімум', 'Максимум', 'Середнє', 'Медіана']
    return stats
stats_07_24 = get_vhi_statistics(df_vhi, province_id=14, years=[2007, 2024])
display(stats_07_24)

--- Порівняльна статистика VHI для області 'Одеська' ---


,Мінімум,Максимум,Середнє,Медіана
Year,,,,
2007,5.52,56.82,36.38,43.29
2024,28.42,68.43,45.87,45.16
